In [ ]:
import numpy as np
from pynq import Overlay, allocate
from IPython.display import Audio
import wave
import struct
import time
ol = Overlay("/home/xilinx/jupyter_notebooks/audio_mix/audio_mix.bit")

In [ ]:
dma = ol.axi_dma_0
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel
mixer = ol.mixer_0

In [ ]:
def load_wav(filename):
    with wave.open(filename, 'rb') as f:
        n_channels = f.getnchannels()
        sampwidth = f.getsampwidth()
        assert sampwidth == 2, f"Expected 16-bit audio, got {sampwidth*8}-bit"
        n_frames = f.getnframes()
        raw = f.readframes(n_frames)
        framerate = f.getframerate()
    
    samples = np.frombuffer(raw, dtype=np.int16)
    
    if n_channels == 2:
        samples = samples[::2]
    
    return samples

def load_and_pack(filenames):
    assert len(filenames) <= 8, "Maximum 8 channels"
    
    channels = [load_wav(f) for f in filenames]
    
    max_len = max(len(c) for c in channels)
    padded = [np.pad(c, (0, max_len - len(c))) for c in channels]
    
    buf = np.zeros((max_len, 8), dtype=np.int16)
    for i, ch in enumerate(padded):
        buf[:, i] = ch

    # 32-bit * 4 = 128 bit wide channel
    return buf.view(np.int32).reshape(max_len * 4), max_len


def stream_wavs(filenames):
    packed, length = load_and_pack(filenames)
    
    CHUNK = 8188
    n_out = CHUNK // 4

    buf = allocate(shape=(CHUNK,), dtype=np.int32)
    output_buffer = allocate(shape=(n_out,), dtype=np.int32)
    all_output = []

    

    for offset in range(0, len(packed), CHUNK):
        chunk = packed[offset:offset+CHUNK]
        if len(chunk) < CHUNK:
            chunk = np.pad(chunk, (0, CHUNK - len(chunk)))

        buf[:] = chunk
        mixer.write(0x0, 0x1)
        dma_recv.transfer(output_buffer)
        dma_send.transfer(buf)
        dma_send.wait()
        dma_recv.wait()

        samples = output_buffer.view(np.int16)[0::2].copy()
        all_output.append(samples)

    del buf
    del output_buffer
    return np.concatenate(all_output)

In [ ]:
output_buffer = stream_wavs([
    "ring_Ringtone_Sample.wav",
    "ring_Drums.wav",
    "ring_Lead_Vox.wav",
    "ring_Piano.wav",
    "ring_Ringtone_Bass.wav",
    "ring_Guitar.wav",
    "ring_Dink.wav"
    ])
dataO = output_buffer.view(np.int16)[0::2].astype(np.float32)
dataO /= 32768.0
del output_buffer
Audio(data=dataO,rate=44100/2)